In [ ]:
# Feb Internship Task 4 - Fine-Tuning BERT
!pip install transformers datasets scikit-learn
import pandas as pd
import re
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments


# Task 1 - Load Dataset
df = pd.read_csv("train.csv")

# reduce size (for faster training)
df = df.sample(5000)


# Task 2 - Preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text

df['clean_text'] = df['tweet'].apply(clean_text)


# Task 3 - Split Data (train, val, test)
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['clean_text'], df['label'], test_size=0.3, random_state=42
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)


# Task 4 - Tokenization
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=64)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=64)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=64)


# Task 5 - Dataset Class
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)
train_dataset = Dataset(train_encodings, train_labels)
val_dataset = Dataset(val_encodings, val_labels)
test_dataset = Dataset(test_encodings, test_labels)


# Task 6 - Model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


# Task 7 - Training (Fine-tuning)
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    learning_rate=2e-5,
    logging_dir="./logs"
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)
trainer.train()


# Task 8 - Evaluation
preds = trainer.predict(test_dataset)
y_pred = preds.predictions.argmax(axis=1)
print("Accuracy:", accuracy_score(test_labels, y_pred))
print("Precision:", precision_score(test_labels, y_pred))
print("Recall:", recall_score(test_labels, y_pred))
print("F1 Score:", f1_score(test_labels, y_pred))
print("Confusion Matrix:\n", confusion_matrix(test_labels, y_pred))


# Task 9 - Experiment (Freeze BERT layers)
for param in model.bert.parameters():
    param.requires_grad = False
print("Experiment: BERT layers frozen")